In [1]:
### Sanity check (MANDATORY)

import sys, numpy, torch
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Torch:", torch.__version__)

Python: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 21:00:12) [Clang 16.0.6 ]
NumPy: 1.26.4
Torch: 2.2.2


In [2]:
####### Imports. ####

import numpy as np
import torch
import torch.nn as nn
import random
import esm
from tqdm.auto import tqdm
import pandas as pd

In [3]:
####### Load & clean peptides #####

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def load_clean(fname):
    seqs = []
    with open(fname) as f:
        for l in f:
            s = l.strip().upper()
            if s and set(s).issubset(VALID_AA):
                seqs.append(s)
    return seqs

pos = load_clean("unique_peptide_pos")
neg = load_clean("unique_peptide_random_neg")
marine = load_clean("unique_peptide_marine")

print("POS:", len(pos))
print("NEG:", len(neg))
print("MARINE:", len(marine))

POS: 1224
NEG: 1224
MARINE: 1224


In [4]:
######### Load ESM2 (LLM USED HERE) #####

model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()

print("ESM2 loaded")

ESM2 loaded


In [5]:
####### ESM2 embedding function (CPU-safe) #####

def esm2_embed(seqs, batch_size=8):
    embs = []
    for i in tqdm(range(0, len(seqs), batch_size)):
        batch = seqs[i:i+batch_size]
        data = [(str(j), s) for j, s in enumerate(batch)]
        _, _, tokens = batch_converter(data)

        with torch.no_grad():
            out = model(tokens, repr_layers=[6])
            reps = out["representations"][6]

        for k, s in enumerate(batch):
            embs.append(
                reps[k, 1:len(s)+1].mean(0).numpy()
            )

        del tokens, out, reps
    return np.vstack(embs)

In [6]:
####### Generate embeddings (ONCE) ######

X_pos = esm2_embed(pos)
X_neg = esm2_embed(neg)
X_mar = esm2_embed(marine)

print(X_pos.shape, X_neg.shape, X_mar.shape)

  0%|          | 0/153 [00:00<?, ?it/s]

  0%|          | 0/153 [00:00<?, ?it/s]

  0%|          | 0/153 [00:00<?, ?it/s]

(1224, 320) (1224, 320) (1224, 320)


In [7]:
######## Create PAIRS for ranking ########

def make_pairs(X_pos, X_neg, n_pairs=None):
    pairs = []
    targets = []

    n_pos = len(X_pos)
    n_neg = len(X_neg)

    if n_pairs is None:
        n_pairs = min(n_pos, n_neg)

    for i in range(n_pairs):
        p = X_pos[i]
        n = X_neg[random.randint(0, n_neg - 1)]

        # p should rank higher than n
        pairs.append((p, n))
        targets.append(1.0)

        # n should rank lower than p
        pairs.append((n, p))
        targets.append(-1.0)

    return pairs, np.array(targets)

pairs, targets = make_pairs(X_pos, X_neg)
print("Total pairs:", len(pairs))

Total pairs: 2448


In [9]:
######### Ranking model (VERY SMALL NN) ####

class RankingHead(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)

In [14]:
########## Train with Margin Ranking Loss ########

device = "cpu"
dim = X_pos.shape[1]

ranker = RankingHead(dim).to(device)
optimizer = torch.optim.Adam(ranker.parameters(), lr=1e-3)
criterion = nn.MarginRankingLoss(margin=1.0)

ranker.train()

for epoch in range(10):
    total_loss = 0.0

    for (a, b), t in zip(pairs, targets):
        a = torch.tensor(a, dtype=torch.float32)
        b = torch.tensor(b, dtype=torch.float32)
        t = torch.tensor([t], dtype=torch.float32)  # <-- FIX

        score_a = ranker(a)
        score_b = ranker(b)

        loss = criterion(score_a, score_b, t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()


    avg_loss = total_loss / len(pairs)
    print(f"Epoch {epoch+1}/10 | Avg loss: {avg_loss:.4f}")
    
    #print(f"Epoch {epoch+1}/10 | Loss: {total_loss:.4f}")

Epoch 1/10 | Avg loss: 0.3597
Epoch 2/10 | Avg loss: 0.2799
Epoch 3/10 | Avg loss: 0.2500
Epoch 4/10 | Avg loss: 0.2385
Epoch 5/10 | Avg loss: 0.2310
Epoch 6/10 | Avg loss: 0.2105
Epoch 7/10 | Avg loss: 0.2083
Epoch 8/10 | Avg loss: 0.1983
Epoch 9/10 | Avg loss: 0.1851
Epoch 10/10 | Avg loss: 0.1796


In [15]:
##### CHECK - Do positives score higher than negatives?

pos_scores = score_embeddings(X_pos, ranker)
neg_scores = score_embeddings(X_neg, ranker)

print("Mean positive score:", pos_scores.mean())
print("Mean negative score:", neg_scores.mean())
print("Median positive score:", np.median(pos_scores))
print("Median negative score:", np.median(neg_scores))


Mean positive score: -0.52715313
Mean negative score: -2.9127607
Median positive score: -0.7297366
Median negative score: -3.0701394


In [17]:
##### Top-k enrichment (THIS IS THE REAL TEST)

all_scores = np.concatenate([pos_scores, neg_scores])
all_labels = np.array([1]*len(pos_scores) + [0]*len(neg_scores))

idx = np.argsort(all_scores)[::-1]

for k in [1, 5, 10]:
    topk = idx[:int(len(idx)*k/100)]
    print(
        f"Top {k}% ACP fraction:",
        all_labels[topk].mean()
    )


Top 1% ACP fraction: 1.0
Top 5% ACP fraction: 0.9918032786885246
Top 10% ACP fraction: 0.9836065573770492


In [ ]:
##### DO the following if want to compute values else directly go to ranking marine peptide section #####

In [18]:
########## Score & RANK marine peptides #####

def score_embeddings(X, model):
    model.eval()
    with torch.no_grad():
        scores = model(torch.tensor(X, dtype=torch.float32))
    return scores.squeeze().numpy()

marine_scores = score_embeddings(X_mar, ranker)

In [20]:
######## Get scores for positives and negatives

pos_scores = score_embeddings(X_pos, ranker)
neg_scores = score_embeddings(X_neg, ranker)

In [21]:
########## Build labels and scores

import numpy as np

scores = np.concatenate([pos_scores, neg_scores])
labels = np.array([1]*len(pos_scores) + [0]*len(neg_scores))

In [22]:
### Compute AUC (THIS IS VALID)

from sklearn.metrics import roc_auc_score

auc = roc_auc_score(labels, scores)
print("AUC (ranking scores):", auc)

AUC (ranking scores): 0.8466565343457645


In [23]:
###### Choose a threshold (THIS MUST BE JUSTIFIED)

thr = np.median(scores)

### Youden’s J (maximizes MCC on validation set)

from sklearn.metrics import matthews_corrcoef

best_mcc, best_thr = -1, None
for t in np.linspace(scores.min(), scores.max(), 200):
    pred = (scores >= t).astype(int)
    mcc = matthews_corrcoef(labels, pred)
    if mcc > best_mcc:
        best_mcc, best_thr = mcc, t

thr = best_thr

In [24]:
#### Compute Acc & MCC

from sklearn.metrics import accuracy_score, matthews_corrcoef

pred = (scores >= thr).astype(int)

acc = accuracy_score(labels, pred)
mcc = matthews_corrcoef(labels, pred)

print("Accuracy:", acc)
print("MCC:", mcc)

Accuracy: 0.7691993464052288
MCC: 0.5389041380835197


In [13]:
######## Save ranked marine peptides ######

df = pd.DataFrame({
    "peptide": marine,
    "LLM_ranking_score": marine_scores
})

df = df.sort_values("LLM_ranking_score", ascending=False)
df.to_csv("marine_peptides_LLM_pairwise_ranked.csv", index=False)

df.head(10)

,peptide,LLM_ranking_score
328,FLSGIVGMLAKLFFLSGIVGMLAKLFFLSGIVGMLAKLFKK,5.369892
23,ALSKALSKALSKALSKALSKALSK,5.262892
133,FAKKLAKLAKKLAKLAL,4.421726
120,FAKKLAKKLAKLL,4.407281
122,FAKKLAKKLKKLAKKLAKKWKL,4.258621
123,FAKKLAKKLKKLAKKLAKLAKKL,4.206216
121,FAKKLAKKLKKLAKKLAK,3.806368
741,KLLKKVVKLFKKLLK,3.678192
933,NFFKRIRRAWKRIWKWIYSA,3.561202
134,FAKKLAKLAKKLAKLALAL,3.533651
